# Lakehouse Monitoring Orchestration (API)

This notebook connects the metadata tables (`monitors_control`, `metric_templates`, `metric_bindings`) 
with the Databricks Lakehouse Monitoring API. 

It reads the metadata, prepares monitor definitions, and ensures that monitors are created or updated 
to reflect the latest configuration.

In [0]:
# ────────────────────────────────────────────────
# Widgets: configure where monitors_control lives
# ────────────────────────────────────────────────
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")
dbutils.widgets.text("out_schema", "lakehouse_monitoring_demo_results", "Output Schema")

# Access values
catalog      = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")
out_schema   = dbutils.widgets.get("out_schema")

print(f"📂 Using catalog={catalog}, admin_schema={admin_schema}, out_schema={out_schema}")

In [0]:
from enum import Enum
import pyspark.sql.functions as F
from pyspark.sql import types as T
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    MonitorMetric, MonitorMetricType, MonitorTimeSeries,
    MonitorCronSchedule, MonitorNotifications
)

w = WorkspaceClient()

### Load Metadata Tables

We always reload the metadata to ensure the latest configuration is used.

In [0]:
ADMIN = f"{catalog}.{admin_schema}"

ctrl_df      = spark.table(f"{ADMIN}.monitors_control").filter("enabled = true")
templates_df = spark.table(f"{ADMIN}.metric_templates")
bindings_df  = spark.table(f"{ADMIN}.metric_bindings").filter("enabled = true")

### Render Metrics from Templates

Each binding + template combination gets expanded into a monitor metric definition.

In [0]:
_METRIC_ENUM = {
    "AGGREGATE": MonitorMetricType.CUSTOM_METRIC_TYPE_AGGREGATE,
    "DERIVED":   MonitorMetricType.CUSTOM_METRIC_TYPE_DERIVED,
    "DRIFT":     MonitorMetricType.CUSTOM_METRIC_TYPE_DRIFT,
}

def _spark_type_json_str(simple: str) -> str:
    t = (simple or "").strip().lower()
    if t == "double":   dt = T.DoubleType()
    elif t == "string": dt = T.StringType()
    elif t == "long":   dt = T.LongType()
    elif t == "boolean":dt = T.BooleanType()
    else:               dt = T.StringType()
    return T.StructField("output", dt).json()

def _render_metric(tmpl_row, bind_row) -> MonitorMetric:
    definition = tmpl_row.definition_template
    params = bind_row.params or {}
    for k,v in params.items():
        definition = definition.replace("{{" + k + "}}", v)
    input_cols = bind_row.input_columns
    if isinstance(input_cols, str):
        input_cols = [input_cols]
    return MonitorMetric(
        type=_METRIC_ENUM[tmpl_row.metric_type.upper()],
        name=bind_row.metric_name,
        input_columns=input_cols,
        definition=definition,
        output_data_type=_spark_type_json_str(tmpl_row.output_spark_type)
    )

def _metrics_for_table(tc, ts, tn):
    b = bindings_df.filter(
        (F.col("table_catalog")==tc)&(F.col("table_schema")==ts)&(F.col("table_name")==tn)
    )
    if b.count()==0: return []
    tmap = {r.template_name:r for r in templates_df.collect()}
    return [_render_metric(tmap[r.template_name], r) for r in b.collect()]

### Change Classification

We classify changes to decide if a monitor needs to be updated (light) or fully refreshed (heavy).

In [0]:
class ChangeClass(Enum):
    NONE   = "none"
    LIGHT  = "light"
    HEAVY  = "heavy"

def _classify_change(existing, desired) -> ChangeClass:
    # compare metrics
    exist_metrics = {(m.name, m.definition) for m in (existing.custom_metrics or [])}
    des_metrics   = {(m.name, m.definition) for m in desired["custom_metrics"]}
    if exist_metrics != des_metrics: return ChangeClass.HEAVY
    
    # time-series
    ex_ts, ds_ts = getattr(existing, "time_series", None), desired.get("time_series")
    if (ex_ts is None) != (ds_ts is None): return ChangeClass.HEAVY
    if ex_ts and ds_ts:
        if ex_ts.timestamp_col != ds_ts.timestamp_col: return ChangeClass.HEAVY
        if set(ex_ts.granularities or []) != set(ds_ts.granularities or []): return ChangeClass.HEAVY
    
    # output schema
    if existing.output_schema_name != desired["output_schema_name"]: return ChangeClass.HEAVY
    
    # schedule
    ex_sched = (getattr(getattr(existing,"schedule",None),"quartz_cron_expression",None),
                getattr(getattr(existing,"schedule",None),"timezone_id",None))
    ds_sched = (None,None)
    if "schedule" in desired:
        ds = desired["schedule"]
        ds_sched = (ds.quartz_cron_expression, ds.timezone_id)
    if ex_sched != ds_sched: return ChangeClass.LIGHT
    
    # notifications
    ex_notif = getattr(existing,"notifications",None)
    ex_emails = set(getattr(getattr(ex_notif,"on_failure",None),"email_addresses",[]) or [])
    des_emails = set(desired.get("notifications",{}).get("on_failure",{}).get("email_addresses",[]))
    if ex_emails != des_emails: return ChangeClass.LIGHT
    
    return ChangeClass.NONE

### Debug Utilities

These functions are useful for inspecting metrics and comparing definitions.

In [0]:
def preview_rendered(table_fqn: str):
    tc, ts, tn = table_fqn.split(".")
    b = bindings_df.filter(
        (F.col("table_catalog")==tc)&(F.col("table_schema")==ts)&(F.col("table_name")==tn)
    ).collect()
    tmap = {r.template_name:r for r in templates_df.collect()}
    return [(row.metric_name, tmpl.definition_template) for row in b for tmpl in [tmap[row.template_name]]]

def diff_defs(existing_metrics, desired_metrics):
    ex = {(m.name, m.definition) for m in (existing_metrics or [])}
    de = set(desired_metrics)
    print("Different?", ex != de)
    print("Missing in monitor:", [d for d in de if d not in ex])
    print("Extra on monitor:",   [e for e in ex if e not in de])

### Create or Update Monitors

Iterate through `monitors_control`, render the desired monitor spec, 
compare with existing, and create/update as needed.

In [0]:
results = []
for r in ctrl_df.collect():
    table_fqn = f"{r.table_catalog}.{r.table_schema}.{r.table_name}"
    metrics = _metrics_for_table(r.table_catalog, r.table_schema, r.table_name)
    if not metrics:
        results.append((table_fqn, "skipped", "no metric_bindings"))
        continue

    desired = dict(
        table_name=table_fqn,
        output_schema_name=r.output_schema_name,
        custom_metrics=metrics
    )
    if r.schedule_cron and r.schedule_tz:
        desired["schedule"] = MonitorCronSchedule(
            quartz_cron_expression=r.schedule_cron, timezone_id=r.schedule_tz
        )
    if r.notifications_on_failure:
        desired["notifications"] = MonitorNotifications(
            on_failure={"email_addresses": list(r.notifications_on_failure)}
        )
    if r.timestamp_col:
        desired["time_series"] = MonitorTimeSeries(
            timestamp_col=r.timestamp_col,
            granularities=list(r.granularities) if r.granularities else ["1 day"]
        )

    try:
        existing = w.quality_monitors.get(table_name=table_fqn)
        change = _classify_change(existing, desired)
        if change == ChangeClass.NONE:
            results.append((table_fqn, "unchanged", "no change, refresh skipped"))
        else:
            w.quality_monitors.update(**desired)
            note = "heavy change → refresh queued" if change==ChangeClass.HEAVY else "light change → no refresh"
            if change == ChangeClass.HEAVY:
                w.quality_monitors.run_refresh(table_name=table_fqn)
            results.append((table_fqn, "updated", note))
    except Exception:
        w.quality_monitors.create(assets_dir=r.assets_dir, **desired)
        w.quality_monitors.run_refresh(table_name=table_fqn)
        results.append((table_fqn, "created", "created → refresh queued"))

for t,a,n in results:
    print(f"• {t} -> {a} | {n}")